In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/100_Round_1 - Final_Annotations.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/200_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_200_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_200_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_200_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['body'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id subreddit                                           title  \
 0  1leplj4  abortion                            First period post MA   
 1  1lepiue  abortion        excruciating pain after at home abortion   
 2  1lea9k0  abortion    I took the first pill and I changed my mind.   
 3  1lemh3e  abortion  Where to get an abortion; Vietnam or Thailand?   
 4  1lemk9s  abortion                  Grief From A Guy’s Perspective   
 
                                                 body          created_utc  \
 0  hello everyone! i made a previous post a few d...  2025-06-18 19:04:52   
 1  hello, so i want to start this off by saying m...  2025-06-18 19:02:03   
 2  I too mifepristone less than two hours ago. Ca...   2025-06-18 6:41:21   
 3  My partner and I found out that we were pregna...  2025-06-18 17:03:26   
 4  I (23M) and my girlfriend (22F) went through w...  2025-06-18 17:06:49   
 
                                                  url  \
 0  https://www.reddit.com/r/abor

In [6]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
test2_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test2_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
test2_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test2_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [7]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(test2_emb_A, test2_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_200_test.json
